# Review Cleaning & Preprocessing

**Input:** `data/raw_review/additional/{nama_desa}_reviews_raw.csv`
**Output:** `data/raw_review/additional/{nama_desa}_reviews.csv`

**Pipeline:**
1. Load raw CSV
2. Dedup + filter bahasa Indonesia
3. Full cleaning: lowercase → emoji → special chars → slang normalization → stopwords → stemming
4. Export cleaned CSV

## 1. Setup & Import

In [1]:
# Jalankan sekali jika belum install
# !pip install Sastrawi

In [2]:
import os
import re
import glob
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

BASE_DIR = os.path.dirname(os.getcwd())
RAW_DIR = os.path.join(BASE_DIR, "data", "raw_review", "additional")
LEXICON_PATH = os.path.join(BASE_DIR, "data", "raw", "colloquial-indonesian-lexicon.csv")

print(f"Raw CSV directory: {RAW_DIR}")
print("Setup OK!")

Raw CSV directory: d:\Kuliah\TA\TA_Notebook\data\raw_review\additional
Setup OK!


## 2. Konfigurasi

In [3]:
# Detect raw CSV files
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "*_raw.csv")))

print("Raw CSV files ditemukan:")
for i, f in enumerate(raw_files):
    df_tmp = pd.read_csv(f)
    print(f"  [{i}] {os.path.basename(f)} — {len(df_tmp)} reviews")

# ============================================================
# EDIT DI SINI — pilih index file, atau None untuk clean semua
# ============================================================
FILE_INDEX = None  # None = clean semua file, atau set angka (misal 0)

if FILE_INDEX is not None:
    files_to_clean = [raw_files[FILE_INDEX]]
else:
    files_to_clean = raw_files

print(f"\nAkan di-clean: {[os.path.basename(f) for f in files_to_clean]}")

Raw CSV files ditemukan:
  [0] osing_reviews_raw.csv — 218 reviews
  [1] pentingsari_reviews_raw.csv — 246 reviews

Akan di-clean: ['osing_reviews_raw.csv', 'pentingsari_reviews_raw.csv']


## 3. Load Resources

In [4]:
# --- Load slang lexicon ---
slang_df = pd.read_csv(LEXICON_PATH, usecols=["slang", "formal"])
slang_df = slang_df.dropna(subset=["slang", "formal"]).drop_duplicates(subset="slang")
SLANG_DICT = dict(zip(slang_df["slang"].str.lower().str.strip(), slang_df["formal"].str.lower().str.strip()))
print(f"Loaded {len(SLANG_DICT)} slang -> formal mappings")

# --- Setup Sastrawi stemmer & stopwords ---
stemmer = StemmerFactory().create_stemmer()
sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())
print(f"Sastrawi stemmer OK, stopwords: {len(sastrawi_stopwords)}")

# --- Custom domain-specific stopwords ---
EXTRA_STOPWORDS = {
    # Kata umum review / Google Maps
    "google", "maps", "map", "review", "reviews", "bintang", "star", "stars",
    # Kata umum wisata
    "desa", "wisata", "tempat", "lokasi", "pergi", "datang", "kunjungi",
    # Filler / particles
    "ya", "yah", "dong", "deh", "sih", "nih", "tuh", "nah", "mah", "loh", "lah",
    "kok", "kan", "pun", "kah", "toh",
    # Kata sambung informal
    "udah", "udh", "aja", "doang", "doank", "seh", "bgt", "gtu", "gitu", "gini",
}
STOPWORDS = sastrawi_stopwords | EXTRA_STOPWORDS
print(f"Total stopwords (Sastrawi + custom): {len(STOPWORDS)}")

Loaded 4330 slang -> formal mappings
Sastrawi stemmer OK, stopwords: 809
Total stopwords (Sastrawi + custom): 840


## 4. Cleaning Functions

**Pipeline:**
1. Lowercase
2. Hapus emoji & unicode symbols
3. Hapus HTML tags, URLs, mentions
4. Hapus special chars & angka
5. Normalisasi slang (colloquial lexicon)
6. Stopwords removal (Sastrawi + custom)
7. Stemming (Sastrawi)

In [5]:
# Regex pattern untuk emoji (compile sekali, pakai berulang)
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002702-\U000027B0"  # dingbats
    "\U000024C2-\U0001F251"  # misc
    "\U0001F900-\U0001F9FF"  # supplemental symbols
    "\U0001FA00-\U0001FA6F"  # chess symbols
    "\U0001FA70-\U0001FAFF"  # symbols extended
    "\U00002600-\U000026FF"  # misc symbols
    "\U0000FE00-\U0000FE0F"  # variation selectors
    "\U0000200D"             # zero width joiner
    "]+",
    flags=re.UNICODE,
)

# Kata-kata umum bahasa Indonesia untuk deteksi bahasa
INDONESIAN_WORDS = {
    "dan", "yang", "di", "ini", "itu", "dengan", "untuk", "dari", "ke",
    "ada", "tidak", "juga", "sudah", "bisa", "akan", "sangat", "sekali",
    "saya", "kami", "kita", "mereka", "dia", "anda", "kamu",
    "bagus", "baik", "kurang", "lebih", "banyak", "sedikit",
    "tempat", "wisata", "desa", "jalan", "rumah", "air",
    "tapi", "karena", "kalau", "jadi", "mau", "lagi", "sih",
    "banget", "bgt", "gak", "ga", "ngga", "nggak", "udah", "aja",
    "nya", "yg", "tp", "krn", "sm", "emang", "mantap",
}


def is_indonesian(text, min_match=2):
    """Cek apakah teks kemungkinan bahasa Indonesia."""
    words = set(str(text).lower().split())
    return len(words.intersection(INDONESIAN_WORDS)) >= min_match


def normalize_slang(text):
    """Normalisasi kata slang/gaul ke formal pakai colloquial lexicon."""
    words = text.split()
    return " ".join(SLANG_DICT.get(w, w) for w in words)


def remove_stopwords(text):
    """Hapus stopwords (Sastrawi + custom)."""
    words = text.split()
    return " ".join(w for w in words if w not in STOPWORDS)


def clean_text(text):
    """
    Full cleaning pipeline:
    1. Lowercase
    2. Hapus emoji & unicode symbols
    3. Hapus HTML tags, URLs, mentions
    4. Hapus special chars & angka (keep huruf + spasi)
    5. Normalisasi slang (colloquial lexicon)
    6. Stopwords removal (Sastrawi + custom)
    7. Stemming (Sastrawi)
    """
    text = str(text).lower()
    text = EMOJI_PATTERN.sub(" ", text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = normalize_slang(text)
    text = remove_stopwords(text)
    text = stemmer.stem(text)
    return text


print("Cleaning functions OK!")
# Quick test
test = "Tempatnya bagusss bgt 😍😍 ga nyesel kesini!!! @wisata123 http://example.com"
print(f"Test input : {test}")
print(f"Test output: {clean_text(test)}")

Cleaning functions OK!
Test input : Tempatnya bagusss bgt 😍😍 ga nyesel kesini!!! @wisata123 http://example.com
Test output: tempat bagus banget sesal kesini


## 5. Apply Cleaning & Filtering

In [6]:
all_cleaned = []

for filepath in files_to_clean:
    fname = os.path.basename(filepath)
    print(f"\n{'='*60}")
    print(f"Processing: {fname}")
    print(f"{'='*60}")
    
    df_raw = pd.read_csv(filepath)
    print(f"Total raw: {len(df_raw)}")
    
    # Nama desa dari kolom atau dari filename
    if "nama desa wisata" in df_raw.columns:
        nama_desa = df_raw["nama desa wisata"].iloc[0]
    else:
        nama_desa = fname.replace("_reviews_raw.csv", "").replace("_", " ").title()
    
    # 1. Hapus duplikat
    df = df_raw.drop_duplicates(subset="review_text").copy()
    print(f"After dedup: {len(df)}")
    
    # 2. Filter bahasa Indonesia (pakai teks asli, sebelum cleaning)
    df["is_indo"] = df["review_text"].apply(is_indonesian)
    df = df[df["is_indo"]].copy()
    print(f"After language filter: {len(df)}")
    
    # 3. Clean text
    print("Cleaning (slang norm + stopwords + stemming)... ", end="", flush=True)
    df["cleaned_review"] = df["review_text"].apply(clean_text)
    print("Done!")
    
    # 4. Hapus review pendek (< 3 kata setelah cleaning)
    df = df[df["cleaned_review"].apply(lambda x: len(x.split()) >= 3)].copy()
    print(f"After short filter (>= 3 words): {len(df)}")
    
    # Set kolom output
    df["review"] = df["review_text"]
    df["nama desa wisata"] = nama_desa
    
    all_cleaned.append(df)
    
    # Preview before/after
    print(f"\n--- Contoh Before vs After ({nama_desa}) ---")
    for _, row in df.head(3).iterrows():
        print(f"\n  ASLI   : {row['review_text'][:120]}")
        print(f"  CLEANED: {row['cleaned_review'][:120]}")

# Gabungkan semua
df_all = pd.concat(all_cleaned, ignore_index=True)
print(f"\n{'='*60}")
print(f"TOTAL: {len(df_all)} cleaned reviews dari {len(files_to_clean)} file(s)")
print(f"{'='*60}")

df_all[["review", "cleaned_review", "rating", "nama desa wisata"]].head(10)


Processing: osing_reviews_raw.csv
Total raw: 218
After dedup: 213
After language filter: 107
Cleaning (slang norm + stopwords + stemming)... Done!
After short filter (>= 3 words): 91

--- Contoh Before vs After (Osing) ---

  ASLI   : Ruang Ganti Acak-acakan,Pintu Toilet Jebol kok gak segera di benahi ?
Minim ruang ganti
  CLEANED: ruang ganti acak acak pintu toilet jebol benah minim ruang ganti

  ASLI   : Sangat bagus kolam renang disini suasananya juga terasa tenang
  CLEANED: bagus kolam renang suasana tenang

  ASLI   : Rumah-rumah lama masih terawat dan beberapa alih fungsi. Penduduknya ramah, tamu boleh mengambil foto rumah mereka.
  CLEANED: rumah rumah awat alih fungsi duduk ramah tamu ambil foto rumah

Processing: pentingsari_reviews_raw.csv
Total raw: 246
After dedup: 246
After language filter: 130
Cleaning (slang norm + stopwords + stemming)... Done!
After short filter (>= 3 words): 121

--- Contoh Before vs After (Pentingsari) ---

  ASLI   : Kearifan lokal yang menyenang

,review,cleaned_review,rating,nama desa wisata
0,"Ruang Ganti Acak-acakan,Pintu Toilet Jebol kok...",ruang ganti acak acak pintu toilet jebol benah...,1,Osing
1,Sangat bagus kolam renang disini suasananya ju...,bagus kolam renang suasana tenang,5,Osing
2,Rumah-rumah lama masih terawat dan beberapa al...,rumah rumah awat alih fungsi duduk ramah tamu ...,4,Osing
3,Gak kaya dulu pertama datang sekarang gak tela...,kayak talu bwrsih bangun tbengkakai kasih kesa...,5,Osing
4,Kotor\nBanyak Nyamuk\nKasihan banget lokasi da...,kotor nyamuk kasihan banget adat kondisi kayak...,1,Osing
5,Parah.\nTiket Dewasa 10rb\nAnak2 5rb\nParkir M...,parah tiket dewasa ribu anak ribu parkir mobil...,1,Osing
6,"Tempat tidak terawat, hanya ada pemandian. Jam...",awat mandi jam operasional jam,4,Osing
7,Desa wisata dgn jajanan dan kebudayaan Osing. ...,jajan budaya osing acara kopi sewu lokasi,5,Osing
8,Pas banget kesana ada acara Kopi Sepuluh Ewu.....,pas banget kesana acara kopi puluh ewu rame ba...,5,Osing
9,"Kamar mandi yang rusak, seperti tak ada perawa...",kamar mandi rusak awat one for all rasa,1,Osing


## 6. Export Cleaned CSV

In [7]:
# Export per desa
for nama_desa in df_all["nama desa wisata"].unique():
    df_desa = df_all[df_all["nama desa wisata"] == nama_desa]
    
    # CSV untuk SA_AutoLabel (review + cleaned_review + nama desa wisata)
    filename = nama_desa.lower().replace(" ", "_") + "_reviews.csv"
    output_path = os.path.join(RAW_DIR, filename)
    df_desa[["review", "cleaned_review", "nama desa wisata"]].to_csv(
        output_path, index=False, encoding="utf-8"
    )
    print(f"Saved {len(df_desa)} reviews -> {output_path}")
    
    # CSV lengkap (dengan rating, author, date)
    full_path = os.path.join(RAW_DIR, nama_desa.lower().replace(" ", "_") + "_reviews_full.csv")
    cols = [c for c in ["review", "cleaned_review", "review_text", "rating", "author", "date", "nama desa wisata"] if c in df_desa.columns]
    df_desa[cols].to_csv(full_path, index=False, encoding="utf-8")
    print(f"Full version -> {full_path}")

print(f"\n{'='*50}")
print("NEXT STEP:")
print("1. Jalankan SA_AutoLabel.ipynb -> auto-label + aspect detection")
print("2. Dashboard otomatis menampilkan desa baru")
print(f"{'='*50}")

Saved 91 reviews -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\osing_reviews.csv
Full version -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\osing_reviews_full.csv
Saved 121 reviews -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\pentingsari_reviews.csv
Full version -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\pentingsari_reviews_full.csv

NEXT STEP:
1. Jalankan SA_AutoLabel.ipynb -> auto-label + aspect detection
2. Dashboard otomatis menampilkan desa baru


## Troubleshooting

### Sastrawi error
- Install: `pip install Sastrawi`
- Method names pakai **snake_case**: `create_stemmer()`, `get_stop_words()`

### Tidak ada `*_raw.csv` ditemukan
- Pastikan sudah jalankan `Scrape_GoogleMaps.ipynb` terlebih dahulu
- Cek folder `data/raw_review/additional/`

### Review terlalu banyak yang ke-filter
- Kurangi `min_match` di `is_indonesian()` (default=2, coba 1)
- Cek STOPWORDS — mungkin ada kata penting yang masuk stopwords

### Stemming terlalu agresif
- Sastrawi kadang over-stem (e.g. "pemandangan" jadi "pandang")
- Kalau ini masalah, bisa comment out baris `text = stemmer.stem(text)` di `clean_text()`